In [74]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from pathlib import Path
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [2]:
spx_data = pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_data/spx_close_return.csv')
spx_data = (
            spx_data
            .drop(['secid', 'return'], axis=1)
            .assign(date = lambda df: pd.to_datetime(df['date']))
            )
spx_data

def ret(df, day):
    df[f'ret_{day}d'] = (df['close'].shift(-day) - df['close']) / df['close'] * 100

for day in [1,5, 7, 10, 15, 22, 44, 66, 132, 252]:
    ret(spx_data, day)

spx_data.drop(columns=['close'], inplace=True)
spx_data

,date,ret_1d,ret_5d,ret_7d,ret_10d,ret_15d,ret_22d,ret_44d,ret_66d,ret_132d,ret_252d
0,2010-01-04,0.311565,1.234786,1.120045,1.521637,-3.602856,-6.167751,0.658435,5.417524,-3.338070,12.259596
1,2010-01-05,0.054552,-0.026396,1.050575,0.133742,-3.433288,-6.188189,0.799810,5.275754,-3.523035,11.762222
2,2010-01-06,0.400127,0.751007,-0.097613,-1.816839,-4.626519,-7.070370,1.152013,5.290466,-6.354539,12.260584
3,2010-01-07,0.288169,0.592981,0.748014,-4.373341,-5.940317,-6.233741,0.726992,6.040169,-6.169801,11.575822
4,2010-01-08,0.174676,-0.781673,-0.606124,-4.209681,-4.872574,-6.711908,0.482978,5.824556,-5.371273,11.049975
...,...,...,...,...,...,...,...,...,...,...,...
3934,2025-08-25,0.413398,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3935,2025-08-26,0.239099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3936,2025-08-27,0.315673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3937,2025-08-28,-0.639817,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
r_1990_2023 = (
    pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_data/par-yield-curve-rates-1990-2023.csv')
    .rename(columns={'Date': 'date', '1 mo': '1mo', '2 mo': '2mo', '3 mo': '3mo', '4 mo': '4mo', '6 mo': '6mo', '1 yr': '1yr', '2 yr': '2yr', '3 yr': '3yr', '5 yr': '5yr', '7 yr': '7yr', '10 yr': '10yr', '20 yr': '20yr', '30 yr': '30yr'})
    .drop(columns=['2yr', '3yr', '5yr', '7yr', '10yr', '20yr', '30yr'])
)
r_2024 = (
    pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_data/daily-treasury-rates-2024.csv')
    .rename(columns={'Date': 'date', '1 Mo': '1mo', '2 Mo': '2mo', '3 Mo': '3mo', '4 Mo': '4mo', '6 Mo': '6mo', '1 Yr': '1yr', '2 Yr': '2yr', '3 Yr': '3yr', '5 Yr': '5yr', '7 Yr': '7yr', '10 Yr': '10yr', '20 Yr': '20yr', '30 Yr': '30yr'})
    .drop(columns=['2yr', '3yr', '5yr', '7yr', '10yr', '20yr', '30yr'])
)
r_2025 = (
    pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_data/daily-treasury-rates-2025.csv')
    .rename(columns={'Date': 'date', '1 Mo': '1mo', '2 Mo': '2mo', '3 Mo': '3mo', '4 Mo': '4mo', '6 Mo': '6mo', '1 Yr': '1yr', '2 Yr': '2yr', '3 Yr': '3yr', '5 Yr': '5yr', '7 Yr': '7yr', '10 Yr': '10yr', '20 Yr': '20yr', '30 Yr': '30yr'})
    .drop(columns=['1.5 Month'])
    .drop(columns=['2yr', '3yr', '5yr', '7yr', '10yr', '20yr', '30yr'])
    )
r_df = (
    pd.concat([r_1990_2023, r_2024, r_2025], axis=0, ignore_index=True)
    .assign(date = lambda df: pd.to_datetime(df['date']))
    .sort_values('date')
    .reset_index(drop=True)
)
r_df

,date,1mo,2mo,3mo,4mo,6mo,1yr
0,1990-01-02,NaN,NaN,7.83,NaN,7.89,7.81
1,1990-01-03,NaN,NaN,7.89,NaN,7.94,7.85
2,1990-01-04,NaN,NaN,7.84,NaN,7.90,7.82
3,1990-01-05,NaN,NaN,7.79,NaN,7.85,7.79
4,1990-01-08,NaN,NaN,7.79,NaN,7.88,7.81
...,...,...,...,...,...,...,...
9001,2025-12-24,3.72,3.74,3.69,3.67,3.59,3.50
9002,2025-12-26,3.70,3.72,3.64,3.66,3.58,3.49
9003,2025-12-29,3.69,3.70,3.68,3.66,3.59,3.48
9004,2025-12-30,3.65,3.65,3.65,3.63,3.59,3.47


In [4]:
excess_ret_df = (
    spx_data.merge(r_df, on='date', how='inner')
    .assign(ret_1d = lambda df: df['ret_1d'] - df['3mo'])
    .assign(ret_1w = lambda df: df['ret_5d'] - df['3mo'])
    .assign(ret_9d = lambda df: df['ret_7d'] - df['3mo'])
    .assign(ret_2w = lambda df: df['ret_10d'] - df['3mo'])
    .assign(ret_3w = lambda df: df['ret_15d'] - df['3mo'])
    .assign(ret_1m = lambda df: df['ret_22d'] - df['3mo'])
    .assign(ret_2m = lambda df: df['ret_44d'] - df['3mo'])
    .assign(ret_3m = lambda df: df['ret_66d'] - df['3mo'])
    .assign(ret_6m = lambda df: df['ret_132d'] - df['3mo'])
    # .assign(ret_1y = lambda df: df['ret_252d'] - df['3mo'])
    .filter(items=['date', 'ret_1d', 'ret_1w', 'ret_9d', 'ret_2w', 'ret_3w', 'ret_1m', 'ret_2m', 'ret_3m', 'ret_6m'])
    .sort_values('date')
    .reset_index(drop=True)
)
excess_ret_df

,date,ret_1d,ret_1w,ret_9d,ret_2w,ret_3w,ret_1m,ret_2m,ret_3m,ret_6m
0,2010-01-04,0.231565,1.154786,1.040045,1.441637,-3.682856,-6.247751,0.578435,5.337524,-3.418070
1,2010-01-05,-0.015448,-0.096396,0.980575,0.063742,-3.503288,-6.258189,0.729810,5.205754,-3.593035
2,2010-01-06,0.340127,0.691007,-0.157613,-1.876839,-4.686519,-7.130370,1.092013,5.230466,-6.414539
3,2010-01-07,0.238169,0.542981,0.698014,-4.423341,-5.990317,-6.283741,0.676992,5.990169,-6.219801
4,2010-01-08,0.124676,-0.831673,-0.656124,-4.259681,-4.922574,-6.761908,0.432978,5.774556,-5.421273
...,...,...,...,...,...,...,...,...,...,...
3907,2025-08-25,-3.876602,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3908,2025-08-26,-4.040901,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3909,2025-08-27,-3.944327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3910,2025-08-28,-4.899817,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
all_vix_df = (
    pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_data/all_vix.csv')
    .assign(date = lambda df: pd.to_datetime(df['Date']))
    .drop(columns=['Date'])
    .sort_values('date')
    .reset_index(drop=True)
)
vix_eret_df = (
    all_vix_df.merge(excess_ret_df, on='date', how='inner')
    .filter(items=['date', 'ret_1d', 'VIX1W', 'ret_1w', 'CBOEVIX9D', 'ret_9d', 'VIX2W', 'ret_2w', 'VIX3W', 'ret_3w', 'CBOEVIX', 'ret_1m', 'VIX2M', 'ret_2m', 'CBOEVIX3M', 'ret_3m', 'CBOEVIX6M', 'ret_6m'])
    .dropna()
    .sort_values('date')
    .reset_index(drop=True)
    # .assign(**{f'{vix}_2': lambda df: df[vix] ** 2 for vix in ['CBOEVIX1D', 'VIX1W', 'CBOEVIX9D', 'VIX2W', 'VIX3W', 'CBOEVIX', 'VIX2M', 'CBOEVIX3M', 'CBOEVIX6M']})
    # .assign(**{f'{vix}_delta': lambda df: df[f'{vix}_2'] - df[f'{vix}_2'].shift(1) for vix in ['CBOEVIX1D', 'VIX1W', 'CBOEVIX9D', 'VIX2W', 'VIX3W', 'CBOEVIX', 'VIX2M', 'CBOEVIX3M', 'CBOEVIX6M']})
)
vix_eret_df

,date,ret_1d,VIX1W,ret_1w,CBOEVIX9D,ret_9d,VIX2W,ret_2w,VIX3W,ret_3w,CBOEVIX,ret_1m,VIX2M,ret_2m,CBOEVIX3M,ret_3m,CBOEVIX6M,ret_6m
0,2011-01-04,0.360709,18.843772,0.196955,16.06,0.927548,19.850429,0.782689,20.174822,1.940775,17.38,3.061858,20.588894,3.782217,20.61,4.423848,23.19,2.904402
1,2011-01-05,-0.352289,19.319286,0.596354,15.57,1.166637,19.771523,0.149841,19.919988,1.660150,17.02,3.188477,20.111353,1.313124,20.05,3.612272,22.78,2.960520
2,2011-01-06,-0.334480,12.361377,0.627957,15.71,1.511891,16.632511,0.595771,18.278048,0.045470,17.40,3.831630,20.218002,2.238821,20.35,3.014423,22.87,2.329884
3,2011-01-07,-0.277633,12.713897,1.569792,15.01,0.679505,17.228448,1.381038,18.489948,1.009823,17.14,3.743602,20.011361,1.817530,20.29,3.234754,22.92,4.203689
4,2011-01-10,0.222514,18.297197,1.840156,15.81,0.677722,19.501473,1.537734,19.886699,2.830114,17.54,3.954745,20.376608,0.804519,20.48,3.375891,22.93,4.267405
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3465,2025-02-13,-4.347195,11.966904,-6.007029,12.31,-6.953543,14.495158,-6.965808,15.754270,-9.979674,15.10,-12.523226,23.698667,-17.951782,17.84,-7.195405,19.11,0.962474
3466,2025-02-14,-4.095504,10.950509,-6.488617,11.56,-6.933289,14.014597,-8.672396,15.672659,-12.518254,14.77,-11.525063,23.533985,-19.981666,17.81,-8.755966,19.19,1.405401
3467,2025-02-18,-4.102300,11.313170,-7.184077,13.20,-8.712404,14.056711,-10.073346,15.836069,-13.435403,15.35,-11.953735,23.504672,-18.073731,17.82,-9.031512,19.32,1.399708
3468,2025-02-19,-4.773420,11.169909,-7.401286,13.06,-7.426676,13.868964,-9.247432,15.982318,-13.207785,15.27,-12.096809,23.158019,-16.844415,17.93,-9.895366,19.41,1.481961


In [67]:
def adj_r2(df, d, vix):
    df = df.copy()
    df[f'{vix}_2'] = df[vix] ** 2
    df[f'{vix}_delta'] = df[f'{vix}_2'] - df[f'{vix}_2'].shift(1)
    # df[f'ret_{d}_lag1'] = df[f'ret_{d}'].shift(1)
    df.dropna(inplace=True)
    x = df[[f'{vix}_delta']]
    y = df[f'{vix}_2']
    model = LinearRegression()
    model.fit(x, y)
    df[f'{vix}_resid'] = y - model.predict(x)

    x_r = sm.add_constant(df[[f'{vix}_delta', f'{vix}_resid']])
    y_r = df[f'ret_{d}']
    model_r = sm.OLS(y_r, x_r).fit()
    
    print(model_r.summary())
    
    return model_r.rsquared_adj * 100

print(adj_r2(vix_eret_df, '1d', 'VIX1W'))

ds=['1d', '1w', '9d', '2w', '3w', '1m', '2m', '3m', '6m']
vix = ['VIX1W', 'CBOEVIX9D', 'VIX2W', 'VIX3W', 'CBOEVIX', 'VIX2M', 'CBOEVIX3M', 'CBOEVIX6M']

# numeric table — for analysis
table_adj_r2_num = pd.DataFrame(columns=[f'ret{d}' for d in ds], index=vix, dtype=float)

# formatted table — for latex
table_adj_r2_fmt = pd.DataFrame(columns=[f'ret{d}' for d in ds], index=vix)

for d in ds:
    for v in vix:
        table_adj_r2_num.loc[v, f'ret{d}'] = adj_r2(vix_eret_df, d, v)

# format after all values computed
for d in ds:
    max_v = table_adj_r2_num[f'ret{d}'].astype(float).idxmax()
    for v in vix:
        val = f"{table_adj_r2_num.loc[v, f'ret{d}']:.2f}"
        if v == max_v:
            val = f"\\textbf{{{val}}}"
        table_adj_r2_fmt.loc[v, f'ret{d}'] = val

# Export to LaTeX
table_adj_r2_fmt.columns = ['R1D', 'R1W', 'R9D', 'R2W', 'R3W', 'R1M', 'R2M', 'R3M', 'R6M']
file_path = Path(f"/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_writing/tables/adj_r2_table.tex")
table_adj_r2_fmt.to_latex(
    file_path,
    column_format="l" + "c"*len(ds),
    escape=False
)

table_adj_r2_fmt

                            OLS Regression Results                            
Dep. Variable:                 ret_1d   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     36.69
Date:                Thu, 25 Jun 2026   Prob (F-statistic):           1.71e-16
Time:                        12:49:20   Log-Likelihood:                -7532.2
No. Observations:                3469   AIC:                         1.507e+04
Df Residuals:                    3466   BIC:                         1.509e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -1.3236      0.036    -36.723      

,R1D,R1W,R9D,R2W,R3W,R1M,R2M,R3M,R6M
VIX1W,2.02,1.14,1.60,2.43,4.10,6.71,8.92,10.81,11.72
CBOEVIX9D,2.22,1.23,1.69,2.54,4.46,6.99,9.43,11.41,12.10
VIX2W,1.78,1.03,1.61,2.41,4.28,6.94,9.73,12.02,12.84
VIX3W,1.28,0.85,1.41,2.07,3.91,6.53,9.84,12.56,13.21
CBOEVIX,2.37,2.13,2.86,3.76,5.78,8.52,11.52,\textbf{13.40},\textbf{13.90}
VIX2M,\textbf{3.70},0.29,-0.02,-0.01,0.74,2.25,5.09,7.67,9.99
CBOEVIX3M,2.59,3.57,4.20,\textbf{4.84},\textbf{7.00},\textbf{9.08},\textbf{11.85},13.31,13.13
CBOEVIX6M,2.92,\textbf{3.72},\textbf{4.24},4.74,6.35,8.08,10.40,11.43,10.50


In [83]:
# 1. scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(vix_eret_df[vix])

# 2. fit PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# 3. PC scores with date
pc_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])], index=vix_eret_df.index)
pc_df.insert(0, 'date', vix_eret_df['date'].values)

# 4. variance explained
variance_df = pd.DataFrame({
    'variance_explained': pca.explained_variance_ratio_,
    'cumulative': pca.explained_variance_ratio_.cumsum()
}, index=[f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))])

# 5. loadings
loadings_df = pd.DataFrame(pca.components_, columns=vix, index=[f'PC{i+1}' for i in range(len(vix))])

print(variance_df)
print(loadings_df)
pc_df

excess_ret_pc_df = (
    excess_ret_df.merge(pc_df, on='date', how='inner')
    .filter(items=['date', 'ret_1d', 'ret_1w', 'ret_9d', 'ret_2w', 'ret_3w', 'ret_1m', 'ret_2m', 'ret_3m', 'ret_6m', 'PC1', 'PC2'])
    .dropna()
    .sort_values('date')
    .reset_index(drop=True)
)


excess_ret_pc_df

     variance_explained  cumulative
PC1            0.927977    0.927977
PC2            0.041160    0.969138
PC3            0.022138    0.991276
PC4            0.003538    0.994814
PC5            0.002955    0.997769
PC6            0.001101    0.998869
PC7            0.000799    0.999668
PC8            0.000332    1.000000
        VIX1W  CBOEVIX9D     VIX2W     VIX3W   CBOEVIX     VIX2M  CBOEVIX3M  \
PC1  0.350269   0.356828  0.360779  0.362566  0.364402  0.335997   0.356102   
PC2 -0.475695  -0.344694 -0.272239 -0.114650 -0.002966  0.312015   0.351717   
PC3 -0.062476  -0.114935  0.039576  0.139543 -0.175783  0.850214  -0.296492   
PC4 -0.360804  -0.412888  0.358656  0.706265  0.019248 -0.251335  -0.009441   
PC5  0.622706  -0.507591  0.158321 -0.011239 -0.466717 -0.016746  -0.071031   
PC6 -0.355486   0.416677  0.537749 -0.167627 -0.495309 -0.018362  -0.209577   
PC7 -0.052983  -0.356771  0.592298 -0.553502  0.332501  0.055464   0.207113   
PC8  0.001792   0.107794 -0.021637  0.050300

,date,ret_1d,ret_1w,ret_9d,ret_2w,ret_3w,ret_1m,ret_2m,ret_3m,ret_6m,PC1,PC2
0,2011-01-04,0.360709,0.196955,0.927548,0.782689,1.940775,3.061858,3.782217,4.423848,2.904402,0.332703,0.054319
1,2011-01-05,-0.352289,0.596354,1.166637,0.149841,1.660150,3.188477,1.313124,3.612272,2.960520,0.217459,-0.039929
2,2011-01-06,-0.334480,0.627957,1.511891,0.595771,0.045470,3.831630,2.238821,3.014423,2.329884,-0.245175,0.511151
3,2011-01-07,-0.277633,1.569792,0.679505,1.381038,1.009823,3.743602,1.817530,3.234754,4.203689,-0.245417,0.489020
4,2011-01-10,0.222514,1.840156,0.677722,1.537734,2.830114,3.954745,0.804519,3.375891,4.267405,0.244169,0.068408
...,...,...,...,...,...,...,...,...,...,...,...,...
3465,2025-02-13,-4.347195,-6.007029,-6.953543,-6.965808,-9.979674,-12.523226,-17.951782,-7.195405,0.962474,-0.964076,0.403071
3466,2025-02-14,-4.095504,-6.488617,-6.933289,-8.672396,-12.518254,-11.525063,-19.981666,-8.755966,1.405401,-1.086229,0.508977
3467,2025-02-18,-4.102300,-7.184077,-8.712404,-10.073346,-13.435403,-11.953735,-18.073731,-9.031512,1.399708,-0.953761,0.429664
3468,2025-02-19,-4.773420,-7.401286,-7.426676,-9.247432,-13.207785,-12.096809,-16.844415,-9.895366,1.481961,-0.975529,0.448668


In [95]:
from pathlib import Path
import pandas as pd
import statsmodels.api as sm

ds = ['1d', '1w', '9d', '2w', '3w', '1m', '2m', '3m', '6m']

# run all regressions
models = {}
for d in ds:
    df = excess_ret_pc_df.dropna().copy()
    x_r = sm.add_constant(df[['PC1', 'PC2']])
    y_r = df[f'ret_{d}']
    models[f'ret_{d}'] = sm.OLS(y_r, x_r).fit()

# format coefficient with stars in math mode
def format_coef(model, var):
    coef = model.params[var]
    pval = model.pvalues[var]
    if pval < 0.01:
        stars = '^{***}'
    elif pval < 0.05:
        stars = '^{**}'
    elif pval < 0.1:
        stars = '^{*}'
    else:
        stars = ''
    return f"${coef:.3f}{stars}$"

# build table manually
rows = ['PC1', 'PC2', 'const', 'N', 'Adj. $R^2$']
table_df = pd.DataFrame(index=rows, columns=[f'ret_{d}' for d in ds])

for d, model in models.items():
    table_df.loc['PC1', d]          = format_coef(model, 'PC1')
    table_df.loc['PC2', d]          = format_coef(model, 'PC2')
    table_df.loc['const', d]        = format_coef(model, 'const')
    table_df.loc['N', d]            = f"${int(model.nobs)}$"
    table_df.loc['Adj. $R^2$', d]   = f"${model.rsquared_adj*100:.2f}$"

# export to latex
table_df.columns = ['R1D', 'R1W', 'R9D', 'R2W', 'R3W', 'R1M', 'R2M', 'R3M', 'R6M']
file_path = Path("/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_writing/tables/pc_table.tex")
table_df.to_latex(file_path, escape=False, column_format="l" + "c" * len(ds))

table_df

,R1D,R1W,R9D,R2W,R3W,R1M,R2M,R3M,R6M
PC1,$0.054^{***}$,$0.115^{***}$,$0.142^{***}$,$0.190^{***}$,$0.285^{***}$,$0.417^{***}$,$0.647^{***}$,$0.824^{***}$,$1.055^{***}$
PC2,$-0.273^{***}$,$-0.147^{*}$,$-0.075$,$-0.129$,$-0.184$,$-0.413^{***}$,$-0.446^{***}$,$-0.747^{***}$,$-1.324^{***}$
const,$-1.323^{***}$,$-1.128^{***}$,$-1.036^{***}$,$-0.900^{***}$,$-0.660^{***}$,$-0.324^{***}$,$0.643^{***}$,$1.627^{***}$,$4.807^{***}$
N,$3470$,$3470$,$3470$,$3470$,$3470$,$3470$,$3470$,$3470$,$3470$
Adj. $R^2$,$0.94$,$1.22$,$1.49$,$2.15$,$3.60$,$6.06$,$8.84$,$11.33$,$11.93$
